# 03 — DA/RT spread analysis

**Question:** what market conditions are associated with large differences between PJM day-ahead and real-time prices?

Spread = RT LMP − DA LMP at PJM Western Hub, hourly. Positive = real-time cleared above day-ahead.

* Window: **Jan 2023 – Jun 2025** (train + validation). Test (2025 H2) and holdout (2026) untouched.
* Regime thresholds (load percentiles, volatility median, extreme-event percentiles) are fitted on **2023–2024 only**.
* `load_regime` uses the **D-1 forecast load** (known before the DA auction); `load_regime_actual` uses actual load (ex-post).
* `rt_vol_7d` = std of hourly RT LMP over the 7 complete days ending D-2 (known at decision time).
* Load forecast error = actual − D-1 09:45 forecast: **ex-post only**, it cannot be a trading input.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.config import load_config
from src.analysis.regimes import add_regimes, eda_frame
from src.analysis import spreads as sp, diagnostics as dg, plots as pl

pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)
pl.apply_style()
cfg = load_config()
FIG = cfg["paths"]["figures"]
full = pd.read_parquet(cfg["paths"]["processed"] / "hourly.parquet")          # holdout file is never loaded
train_mask = full["period"] == "train"
full, thresholds = add_regimes(full, train_mask)       # thresholds fitted on 2023-2024 only
full, ev_thr = dg.extreme_flags(full, train_mask)      # extreme-event thresholds fitted on 2023-2024 only
d = eda_frame(full)                                    # train + validation: Jan 2023 - Jun 2025
d["year"] = d["market_date"].dt.year
d["err_pct"] = d["load_forecast_error"] / d["forecast_load"] * 100
print(f"EDA window: {d.market_date.min().date()} -> {d.market_date.max().date()}, {len(d):,} hours")
print("forecast-load regime cut points (MW, 25/75/95th pct of 2023-24):", thresholds["forecast_load_MW"].round(0))
print("extreme thresholds ($/MWh):", {k: round(float(v), 1) for k, v in ev_thr.items()})

## Q1. What does the spread distribution look like?

In [ ]:
sp.distribution_summary(d["da_rt_spread"]).to_frame("2023-01 → 2025-06")

In [ ]:
m, lo, hi = sp.daily_block_bootstrap_mean_ci(d)
print(f"mean spread ${m:.2f}/MWh, 95% CI (resampling whole days) [{lo:.2f}, {hi:.2f}]")
sp.group_stats(d, "year")

In [ ]:
pl.spread_histogram(d["da_rt_spread"], FIG / "01_spread_histogram.png");

In [ ]:
pl.spread_cdf(d["da_rt_spread"], FIG / "02_spread_cdf.png");

In [ ]:
by_hour = sp.group_stats(d, "hour_ending")
pl.profile_band(by_hour, "Hour ending (EPT)", "The spread is widest in the evening peak",
    "Hourly spread by hour of day: median, mean and 5th–95th percentile band.", FIG / "03_spread_by_hour.png")
by_hour

In [ ]:
by_month = sp.group_stats(d, "month")
pl.profile_band(by_month, "Month", "Summer and January carry the widest spreads",
    "Hourly spread by calendar month (Jan 2023 – Jun 2025): median, mean and 5th–95th percentile band.",
    FIG / "04_spread_by_month.png", xticklabels=["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
by_month

## Q2. When are spreads most volatile?

In [ ]:
for by in ("is_weekend", "season", "load_regime", "load_regime_actual", "vol_regime"):
    display(sp.group_stats(d, by))

In [ ]:
heat = d.pivot_table(index="season", columns="hour_ending", values="da_rt_spread", aggfunc="std", observed=True)
pl.volatility_heatmap(heat, "Spread volatility concentrates in summer evenings and winter mornings",
    "Standard deviation of the hourly spread ($/MWh) by season and hour ending (EPT). Top-decile cells labelled.",
    FIG / "05_volatility_heatmap.png")
heat.round(0)

In [ ]:
reg = sp.group_stats(d, "load_regime")
pl.range_dots(reg, "Spread risk rises sharply with forecast load",
    "Spread by D-1 forecast-load regime (25th / 75th / 95th percentile of 2023–24 load). Dot = median, bar = 5th–95th percentile.",
    FIG / "06_spread_by_load_regime.png", xlabel="RT − DA ($/MWh)")
reg

In [ ]:
d.pivot_table(index="season", columns="load_regime", values="da_rt_spread", aggfunc=["std", "median", "count"], observed=True).round(1)

## Q3. Does load forecast error matter?

Load error = actual − D-1 forecast. Hourly observations are autocorrelated, so p-values overstate significance; the bucketed view and the day-level correlation are the more honest evidence.

In [ ]:
d["abs_spread"], d["abs_err"] = d["da_rt_spread"].abs(), d["load_forecast_error"].abs()
pd.DataFrame({
    "error vs spread": dg.correlations(d, "load_forecast_error", "da_rt_spread"),
    "|error| vs |spread|": dg.correlations(d, "abs_err", "abs_spread"),
    "daily mean error vs daily mean spread": dg.correlations(
        d.groupby("market_date").agg(e=("load_forecast_error", "mean"), s=("da_rt_spread", "mean")).dropna(), "e", "s"),
}).T

In [ ]:
# stability: does the relationship hold in each year separately?
pd.DataFrame({y: dg.correlations(g, "load_forecast_error", "da_rt_spread") for y, g in d.groupby("year")}).T[["n", "pearson_r", "spearman_rho"]]

In [ ]:
dg.bucket_table(d, "load_forecast_error", q=10, extreme_flag="ev_rt_high")

In [ ]:
edges = np.array([-20, -4, -2, -1, 0, 1, 2, 4, 20])
err_tbl = dg.bucket_table(d, "err_pct", edges=edges, extreme_flag="ev_rt_high")
labels = ["< −4%", "−4 to −2", "−2 to −1", "−1 to 0", "0 to 1", "1 to 2", "2 to 4", "> +4%"]
pl.error_bucket_panels(err_tbl, labels, "When load beats the forecast, real-time tends to clear above day-ahead",
    "Hourly spread by load forecast error (actual vs D-1 09:45 forecast). The error is known only after the fact — explanatory, not tradable.",
    FIG / "07_forecast_error_vs_spread.png")
err_tbl

In [ ]:
dg.bucket_table(d, "abs_err", q=5, extreme_flag="ev_abs_spread")

In [ ]:
d["err_sign"] = np.where(d["load_forecast_error"] > 0, "actual > forecast", "actual < forecast")
pd.concat({
    "mean spread": d.pivot_table(index="load_regime", columns="err_sign", values="da_rt_spread", aggfunc="mean", observed=True),
    "P(top-1% RT)": d.pivot_table(index="load_regime", columns="err_sign", values="ev_rt_high", aggfunc="mean", observed=True),
    "hours": d.pivot_table(index="load_regime", columns="err_sign", values="da_rt_spread", aggfunc="size", observed=True),
}, axis=1).round(3)

## Q4. What happens during extreme price events?

Thresholds (from 2023–24): top 1% RT LMP, bottom 1% RT LMP, top 1% |spread| (split by sign).

In [ ]:
events = {"Top 1% RT": "ev_rt_high", "Bottom 1% RT": "ev_rt_low",
          "Top 1% |spread|, RT>DA": "ev_spread_pos", "Top 1% |spread|, RT<DA": "ev_spread_neg"}
dg.event_profile(d, events, ["da_lmp", "rt_lmp", "da_rt_spread", "forecast_load", "actual_load",
                             "load_forecast_error", "err_pct", "rt_vol_7d", "hour_ending"])

In [ ]:
d["hour_block"] = pd.cut(d["hour_ending"], [0, 6, 10, 16, 21, 24],
                         labels=["HE1–6", "HE7–10", "HE11–16", "HE17–21", "HE22–24"])
for flag in ("ev_spread_pos", "ev_spread_neg", "ev_rt_high", "ev_rt_low"):
    under = (d.loc[d[flag], "load_forecast_error"] > 0).mean()
    print(f"==== {flag}: {int(d[flag].sum())} hours | actual > forecast in {under:.0%} (all hours: {(d['load_forecast_error'] > 0).mean():.0%})")
    display(pd.concat({by: dg.composition_lift(d, flag, by)["lift"] for by in ("season", "load_regime", "vol_regime", "is_weekend", "hour_block")}).to_frame("lift").T)

In [ ]:
panels = {}
for by, name in (("load_regime", "D-1 forecast-load regime"), ("hour_block", "Hour block (EPT)"), ("season", "Season")):
    panels[name] = pd.DataFrame({"pos": dg.composition_lift(d, "ev_spread_pos", by)["lift"],
                                 "neg": dg.composition_lift(d, "ev_spread_neg", by)["lift"]})
pl.lift_panels(panels, "Extreme spreads are a high-load, evening phenomenon — in both directions",
    "Over-representation of top-1% |spread| hours by category (lift = share of event hours ÷ share of all hours).",
    FIG / "08_extreme_spread_lift.png");

In [ ]:
for flag in ("ev_spread_pos", "ev_spread_neg"):
    days = dg.event_days(d, flag)
    print(f"{flag}: {len(days)} event days; the 5 largest days hold {days.hours.head(5).sum()} of {days.hours.sum()} event hours")
    display(days.head(8))

## Findings

Full write-up: `reports/phase2_findings.md`. In short (Jan 2023 – Jun 2025, associations not causes):

1. **RT usually clears below DA, but rare spikes offset it.** Median spread −$2.58, RT < DA in 67% of hours, every year. The mean of −$0.59 is not distinguishable from zero (95% CI −$1.42 to +$0.21, resampling whole days). Skewness is 18, and the top 1% of hours supply 32% of positive spread dollars.
2. **Volatility is concentrated in predictable places:** evening peak HE18–20 (std $45–63 vs $9–15 overnight), worst in **summer HE19–20 (≈ $117)**, with a winter/spring HE8 morning-ramp peak, and on weekdays.
3. **Forecast load (ex-ante) predicts risk, not direction.** Spread std goes from $9.7 (Low load) to **$93 (Extreme, top 5%)**, while the share of RT > DA hours stays at 32–36% in every regime. The median falls from −$1.6 to −$8.5 as load rises.
4. **Load forecast error is associated with the spread's direction, monotonically and stably** (Spearman 0.24 hourly / 0.32 daily; 0.23, 0.23 and 0.26 by year). Load > 4% above forecast: RT > DA in 66% of hours. Load > 4% below: 13%. The effect is strongest at Extreme load (mean +$27 when under-forecast). This is ex-post information, not tradable as-is.
5. **Extreme spreads are a high-load, evening phenomenon in both directions.** Extreme RT > DA hours are 6.7× over-represented at Extreme load and 59% under-forecast (e.g. the June 2025 heat wave, +$1,477). Extreme RT < DA hours occur only at High/Extreme load and are 86% over-forecast: DA priced scarcity that did not arrive (e.g. 20 Jan 2025). Five days hold 50 of those 84 hours.

**Hypotheses for Phases 3–5:** use forecast load and hour as risk conditioners; test a short-spread base-rate benchmark outside high-risk hours, net of costs; look for ex-ante proxies of forecast error (recent forecast bias, forecast change vs prior day, recent spread behaviour).